### Modifying dt

'trial_' prefix helpful to differentiate datatree child nodes that have trial data vs added once. But awkward do modifications, e.g. here `dt["trial_sample_trial"].`.

In [ ]:
import ethograph as eto
path = r"C:\Users\aksel\Documents\Code\EthoGraph\data\copExpBP08\birdpark_accelerometer.nc"

dt = eto.open(path)
dt.load() 
dt["trial_sample_trial"] = dt.itrial(0).sel(time=slice(20, 25))

dt["trial_sample_trial"].attrs["mic1"]  = "copExpBP08_trim.wav"
dt["trial_sample_trial"].attrs["cam1"]  = "copExpBP08_trim.mp4"

### Labels

I/0 for managing labels, how to improve. Allow loading labels files from other platforms: Boris, NWB, DLC2Action, ...

For audio data, manage file types in https://crowsetta.readthedocs.io/ format.

In [ ]:
ds["labels"] = (("time", "individuals"), np.zeros((len(ds.coords["time"]), len(ds.coords["individuals"]))))

### `time_XX`

Probably not of an interest of movement to work with data not at sample rate of video frame rate??

### File integrity, saving data

Sometimes got corrupted, problem with saving and permission errors

### Relative file paths

Manage paths via xarray attributes. 

In [ ]:
help(eto.set_media_attrs)

### Multiple time coord, fps, offset

In [ ]:
path = r"C:\Users\aksel\Desktop\temp\ibl.nwb" # add external file link

from pynwb import NWBHDF5IO


import pynwb
import xarray as xr
from movement.io import load_poses
with pynwb.NWBHDF5IO(path, mode="r") as io:
    nwb_file = io.read()
    nwb_file
    
    
from movement.napari.convert import ds_to_napari_layers
import napari
io = pynwb.NWBHDF5IO(path, mode="r")
nwb_file = io.read()
ds = load_poses.from_nwb_file(nwb_file, processing_module_key="pose_estimation", pose_estimation_key="RightCamera")


kp_coord = ds.coords.get("keypoints")
keypoints = kp_coord.values.astype(str).tolist()
data, _, properties = ds_to_napari_layers(ds)

viewer = napari.current_viewer() or napari.Viewer()

viewer.add_points(
    data,
    name="pose_keypoints",
    properties=properties,
    size=4,
    opacity=0.9,
    face_color="keypoints" if "keypoints" in properties else "white",
    edge_color="white",
    edge_width=0.2,
)



In [4]:
data

array([[0.00000000e+00, 0.00000000e+00, 2.96157745e+02, 4.11489868e+02],
       [0.00000000e+00, 1.00000000e+00, 2.96158203e+02, 4.11473572e+02],
       [0.00000000e+00, 2.00000000e+00, 2.96165649e+02, 4.11431335e+02],
       ...,
       [1.00000000e+01, 6.03438000e+05, 1.29376633e+02, 4.98010332e+02],
       [1.00000000e+01, 6.03439000e+05, 1.29376633e+02, 4.98010332e+02],
       [1.00000000e+01, 6.03440000e+05, 1.29376633e+02, 4.98010332e+02]],
      shape=(6637851, 4))

### Wrapper from nwb to data trees

Current nwb from path, assumes you ahve the file locally, but NWB files are very large to download indivdiually. Ability to download/stream only the necesary bits via movement functionality.

videos/pose ndx, which camera belongs to which pose situation? Whether it's possible to make ndx-pose more stringent so its easier tog et video data